In [1]:
# Cell 1: imports
import math
from dataclasses import dataclass

import numpy as np
import warp as wp
from tqdm import tqdm


import newton
import newton.examples
import newton.viewer
import trimesh

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 4090" (23 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/bg/.cache/warp/1.12.1


In [2]:
builder = newton.ModelBuilder()

In [22]:
def add_cylindrical_container(
    builder,
    radius,
    height,
    static_cfg,
    wall_thickness=0.3,
    floor_thickness=0.25,
    n_faces=32,
    center=(0.0, 0.0, 0.0),
):
    """
    Create an open-top cylindrical container approximated by box wall segments.

    Newton convention: x, y, z

    Args:
        builder: Newton ModelBuilder
        radius (float): inner radius of the container
        height (float): wall height along Z
        static_cfg: ShapeConfig for static objects
        wall_thickness (float): radial wall thickness
        floor_thickness (float): floor thickness
        n_faces (int): number of wall segments
        center (tuple): (x, y, z) centre of the container base
    """

    if n_faces < 3:
        raise ValueError("n_faces must be at least 3")

    cx, cy, cz = center
    static_body = -1

    # -----------------------------
    # Floor
    # -----------------------------
    builder.add_shape_cylinder(
        static_body,
        radius=radius,
        half_height=floor_thickness * 0.5,
        xform=wp.transform(
            p=wp.vec3(cx, cy, cz + floor_thickness * 0.5),
            q=wp.quat_identity(),
        ),
        cfg=static_cfg,
    )

    # -----------------------------
    # Wall segments
    # -----------------------------
    angle_step = 2.0 * math.pi / n_faces

    # Segment chord length around the inner radius
    segment_length = 2.0 * radius * math.tan(math.pi / n_faces)

    for i in range(n_faces):
        theta = i * angle_step

        # Wall centre is halfway through the wall thickness
        wall_radius = radius + wall_thickness * 0.5

        x = cx + wall_radius * math.cos(theta)
        y = cy + wall_radius * math.sin(theta)

        # Rotate each box so its long side is tangent to the circle
        q = wp.quat_from_axis_angle(
            wp.vec3(0.0, 0.0, 1.0),
            theta,
        )

        builder.add_shape_box(
            static_body,
            hx=wall_thickness * 0.5,
            hy=segment_length * 0.5,
            hz=height * 0.5,
            xform=wp.transform(
                p=wp.vec3(x, y, cz + height * 0.5),
                q=q,
            ),
            cfg=static_cfg,
        )

def add_particle(
    builder,
    path,
    position,
    orientation,
    scale=1.0,
    dynamic_cfg=None,
):
    """
    Add an OBJ particle mesh to a Newton ModelBuilder.

    Collision uses the convex hull of the OBJ mesh.

    Parameters
    ----------
    builder : newton.ModelBuilder
    path : str
        Path to .obj file.
    position : tuple/list/wp.vec3
        Newton uses (x, y, z).
    orientation : tuple/list/wp.quat
        Quaternion as (x, y, z, w), or wp.quat.
    scale : float
    dynamic_cfg : newton.ModelBuilder.ShapeConfig | None
        If None, a dynamic config is created.

    Returns
    -------
    body_id : int
    """

    # -----------------------------
    # Load OBJ mesh
    # -----------------------------
    mesh = trimesh.load(path, force="mesh")

    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected a trimesh.Trimesh, got {type(mesh)}")

    # Apply scale
    mesh = mesh.copy()
    vertices = mesh.vertices
    com = vertices.mean(axis=0)
    vertices_centered = vertices - com
    mesh.vertices = vertices_centered
    mesh.apply_scale(scale)

    # -----------------------------
    # Use convex hull for collision
    # -----------------------------
    hull = mesh.convex_hull

    vertices = np.asarray(hull.vertices, dtype=np.float32)
    faces = np.asarray(hull.faces, dtype=np.int32)

    if vertices.shape[0] == 0 or faces.shape[0] == 0:
        raise ValueError("Convex hull mesh is empty.")

    # Newton expects flattened triangle indices
    indices = faces.reshape(-1).astype(np.int32)

    collision_mesh = newton.Mesh(
        vertices,
        indices,
        compute_inertia=True,
        is_solid=True,
    )

    # -----------------------------
    # Shape config
    # -----------------------------
    if dynamic_cfg is None:
        cfg = newton.ModelBuilder.ShapeConfig()
        cfg.density = 1000.0
        cfg.ke = 1.0e5
        cfg.kd = 1.0e-4
        cfg.mu = 0.6
        cfg.restitution = 0.35
    else:
        cfg = dynamic_cfg

    # -----------------------------
    # Transform
    # -----------------------------
    if isinstance(position, wp.vec3):
        p = position
    else:
        p = wp.vec3(float(position[0]), float(position[1]), float(position[2]))

    if isinstance(orientation, wp.quat):
        q = orientation
    else:
        q = wp.quat(
            float(orientation[0]),
            float(orientation[1]),
            float(orientation[2]),
            float(orientation[3]),
        )

    # -----------------------------
    # Dynamic body
    # -----------------------------
    body_id = builder.add_body(
        xform=wp.transform(p=p, q=q),
        label="particle",
    )

    builder.add_shape_mesh(
        body=body_id,
        mesh=collision_mesh,
        cfg=cfg,
    )

    return body_id

In [4]:
static_cfg = newton.ModelBuilder.ShapeConfig()
static_cfg.ke = 1.0e5
static_cfg.kd = 1.0e-4
static_cfg.mu = 0.8
static_cfg.restitution = 0.2

dynamic_cfg = newton.ModelBuilder.ShapeConfig()
dynamic_cfg.density = 1000.0
dynamic_cfg.ke = 1.0e5
dynamic_cfg.kd = 1.0e-4
dynamic_cfg.mu = 0.6
dynamic_cfg.restitution = 0.35

In [5]:
add_cylindrical_container(
    builder,
    radius=1.0,
    height=3.0,
    wall_thickness=0.3,
    floor_thickness=0.25,
    n_faces=32,
    center=(0.0, 0.0, 0.0),
    static_cfg=static_cfg,
)

In [6]:
add_particle(
    builder,
    '/home/bg/Developer/particle-pack-generation/data/toy_data_GOH_6_250/tmp/830.obj',
    position=(0.0, 0.0, 10.0),
    orientation=(0.0, 0.0, 0.0, 1.0),
    scale=1.0,
    dynamic_cfg=dynamic_cfg
)

Module newton._src.geometry.inertia 5855353 load on device 'cuda:0' took 0.77 ms  (cached)


0

In [9]:
viewer = newton.viewer.ViewerRerun()

# -----------------------------
# Simulation settings
# -----------------------------
frame_dt = 1.0 / 60.0
sim_substeps = 10
sim_dt = frame_dt / sim_substeps
sim_time = 0.0
num_frames = 600


# -----------------------------
# Finalise model
# -----------------------------
model = builder.finalize()

collision_pipeline = newton.CollisionPipeline(
    model,
    broad_phase="sap",
)

solver = newton.solvers.SolverXPBD(
    model,
    iterations=12,
    rigid_contact_relaxation=0.8,
)

state_0 = model.state()
state_1 = model.state()
control = model.control()

newton.eval_fk(
    model,
    model.joint_q,
    model.joint_qd,
    state_0,
)

contacts = collision_pipeline.contacts()

# -----------------------------
# Viewer setup
# -----------------------------
viewer.set_model(model)

# -----------------------------
# Viewer setup
# -----------------------------
viewer.set_model(model)

viewer.set_camera(
    pos=wp.vec3(10.0, -14.0, 10.0),
    pitch=-30.0,
    yaw=135.0,
)


# -----------------------------
# Simulation loop
# -----------------------------
for frame in tqdm(range(num_frames)):
    for _ in range(sim_substeps):
        state_0.clear_forces()

        contacts = model.collide(
            state_0,
            collision_pipeline=collision_pipeline,
        )

        viewer.apply_forces(state_0)

        solver.step(
            state_0,
            state_1,
            control,
            contacts,
            sim_dt,
        )

        state_0, state_1 = state_1, state_0

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()

    sim_time += frame_dt
viewer.show_notebook(width=1600, height=800)

[2026-04-29T06:19:38Z WARN  re_sdk::log_sink] Dropping data in BufferedSink
100%|██████████| 600/600 [00:09<00:00, 64.55it/s]


HTML(value='<div id="9dd3f96e-5e6f-4425-a0a2-d8c95b5d4915"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

In [10]:
builder = newton.ModelBuilder()
def add_cylindrical_container(
    builder,
    radius,
    height,
    static_cfg,
    wall_thickness=0.3,
    floor_thickness=0.25,
    n_faces=32,
    center=(0.0, 0.0, 0.0),
):
    """
    Create an open-top cylindrical container approximated by box wall segments.

    Newton convention: x, y, z

    Args:
        builder: Newton ModelBuilder
        radius (float): inner radius of the container
        height (float): wall height along Z
        static_cfg: ShapeConfig for static objects
        wall_thickness (float): radial wall thickness
        floor_thickness (float): floor thickness
        n_faces (int): number of wall segments
        center (tuple): (x, y, z) centre of the container base
    """

    if n_faces < 3:
        raise ValueError("n_faces must be at least 3")

    cx, cy, cz = center
    static_body = -1

    # -----------------------------
    # Floor
    # -----------------------------
    builder.add_shape_cylinder(
        static_body,
        radius=radius,
        half_height=floor_thickness * 0.5,
        xform=wp.transform(
            p=wp.vec3(cx, cy, cz + floor_thickness * 0.5),
            q=wp.quat_identity(),
        ),
        cfg=static_cfg,
    )

    # -----------------------------
    # Wall segments
    # -----------------------------
    angle_step = 2.0 * math.pi / n_faces

    # Segment chord length around the inner radius
    segment_length = 2.0 * radius * math.tan(math.pi / n_faces)

    for i in range(n_faces):
        theta = i * angle_step

        # Wall centre is halfway through the wall thickness
        wall_radius = radius + wall_thickness * 0.5

        x = cx + wall_radius * math.cos(theta)
        y = cy + wall_radius * math.sin(theta)

        # Rotate each box so its long side is tangent to the circle
        q = wp.quat_from_axis_angle(
            wp.vec3(0.0, 0.0, 1.0),
            theta,
        )

        builder.add_shape_box(
            static_body,
            hx=wall_thickness * 0.5,
            hy=segment_length * 0.5,
            hz=height * 0.5,
            xform=wp.transform(
                p=wp.vec3(x, y, cz + height * 0.5),
                q=q,
            ),
            cfg=static_cfg,
        )

def add_particle(
    builder,
    path,
    position,
    orientation,
    scale=1.0,
    dynamic_cfg=None,
):
    """
    Add an OBJ particle mesh to a Newton ModelBuilder.

    Collision uses the convex hull of the OBJ mesh.

    Parameters
    ----------
    builder : newton.ModelBuilder
    path : str
        Path to .obj file.
    position : tuple/list/wp.vec3
        Newton uses (x, y, z).
    orientation : tuple/list/wp.quat
        Quaternion as (x, y, z, w), or wp.quat.
    scale : float
    dynamic_cfg : newton.ModelBuilder.ShapeConfig | None
        If None, a dynamic config is created.

    Returns
    -------
    body_id : int
    """

    # -----------------------------
    # Load OBJ mesh
    # -----------------------------
    mesh = trimesh.load(path, force="mesh")

    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected a trimesh.Trimesh, got {type(mesh)}")

    # Apply scale
    mesh = mesh.copy()
    mesh.apply_scale(scale)

    # -----------------------------
    # Use convex hull for collision
    # -----------------------------
    hull = mesh.convex_hull

    vertices = np.asarray(hull.vertices, dtype=np.float32)
    faces = np.asarray(hull.faces, dtype=np.int32)

    if vertices.shape[0] == 0 or faces.shape[0] == 0:
        raise ValueError("Convex hull mesh is empty.")

    # Newton expects flattened triangle indices
    indices = faces.reshape(-1).astype(np.int32)

    collision_mesh = newton.Mesh(
        vertices,
        indices,
        compute_inertia=True,
        is_solid=True,
    )

    # -----------------------------
    # Shape config
    # -----------------------------
    if dynamic_cfg is None:
        cfg = newton.ModelBuilder.ShapeConfig()
        cfg.density = 1000.0
        cfg.ke = 1.0e5
        cfg.kd = 1.0e-4
        cfg.mu = 0.6
        cfg.restitution = 0.35
    else:
        cfg = dynamic_cfg

    # -----------------------------
    # Transform
    # -----------------------------
    if isinstance(position, wp.vec3):
        p = position
    else:
        p = wp.vec3(float(position[0]), float(position[1]), float(position[2]))

    if isinstance(orientation, wp.quat):
        q = orientation
    else:
        q = wp.quat(
            float(orientation[0]),
            float(orientation[1]),
            float(orientation[2]),
            float(orientation[3]),
        )

    # -----------------------------
    # Dynamic body
    # -----------------------------
    body_id = builder.add_body(
        xform=wp.transform(p=p, q=q),
        label="particle",
    )

    builder.add_shape_mesh(
        body=body_id,
        mesh=collision_mesh,
        cfg=cfg,
    )

    return body_id
static_cfg = newton.ModelBuilder.ShapeConfig()
static_cfg.ke = 1.0e5
static_cfg.kd = 1.0e-4
static_cfg.mu = 0.8
static_cfg.restitution = 0.2

dynamic_cfg = newton.ModelBuilder.ShapeConfig()
dynamic_cfg.density = 1000.0
dynamic_cfg.ke = 1.0e5
dynamic_cfg.kd = 1.0e-4
dynamic_cfg.mu = 0.6
dynamic_cfg.restitution = 0.35
add_cylindrical_container(
    builder,
    radius=1.0,
    height=3.0,
    wall_thickness=0.3,
    floor_thickness=0.25,
    n_faces=32,
    center=(0.0, 0.0, 0.0),
    static_cfg=static_cfg,
)
add_particle(
    builder,
    '/home/bg/Developer/particle-pack-generation/data/toy_data_GOH_6_250/tmp/830.obj',
    position=(0.0, 0.0, 2.0),
    orientation=(0.0, 0.0, 0.0, 1.0),
    scale=0.5,
    dynamic_cfg=dynamic_cfg
)
viewer = newton.viewer.ViewerRerun()

# -----------------------------
# Simulation settings
# -----------------------------
frame_dt = 1.0 / 60.0
sim_substeps = 10
sim_dt = frame_dt / sim_substeps
sim_time = 0.0
num_frames = 600


# -----------------------------
# Finalise model
# -----------------------------
model = builder.finalize()

collision_pipeline = newton.CollisionPipeline(
    model,
    broad_phase="sap",
)

solver = newton.solvers.SolverXPBD(
    model,
    iterations=12,
    rigid_contact_relaxation=0.8,
)

state_0 = model.state()
state_1 = model.state()
control = model.control()

newton.eval_fk(
    model,
    model.joint_q,
    model.joint_qd,
    state_0,
)

contacts = collision_pipeline.contacts()

# -----------------------------
# Viewer setup
# -----------------------------
viewer.set_model(model)

viewer.set_camera(
    pos=wp.vec3(10.0, -14.0, 10.0),
    pitch=-30.0,
    yaw=135.0,
)


# -----------------------------
# Simulation loop
# -----------------------------
for frame in tqdm(range(num_frames)):
    for _ in range(sim_substeps):
        state_0.clear_forces()

        contacts = model.collide(
            state_0,
            collision_pipeline=collision_pipeline,
        )

        viewer.apply_forces(state_0)

        solver.step(
            state_0,
            state_1,
            control,
            contacts,
            sim_dt,
        )

        state_0, state_1 = state_1, state_0

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()

    sim_time += frame_dt
viewer.show_notebook()

100%|██████████| 600/600 [00:09<00:00, 63.46it/s]


HTML(value='<div id="6d0013ea-ad25-4173-8c12-38692862deb8"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

In [17]:
# -----------------------------
# Viewer
# -----------------------------
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)

# -----------------------------
# Simulation settings
# -----------------------------
frame_dt = 1.0 / 60.0
sim_substeps = 10
sim_dt = frame_dt / sim_substeps
sim_time = 0.0
num_frames = 600

# -----------------------------
# Build scene
# -----------------------------
builder = newton.ModelBuilder()

# -----------------------------
# Materials
# -----------------------------
dynamic_cfg = newton.ModelBuilder.ShapeConfig()
dynamic_cfg.ke = 1.0e5
dynamic_cfg.kd = 1.0e-4
dynamic_cfg.mu = 0.8
dynamic_cfg.restitution = 0.2

dynamic_cfg = newton.ModelBuilder.ShapeConfig()
dynamic_cfg.density = 1000.0
dynamic_cfg.ke = 1.0e5
dynamic_cfg.kd = 1.0e-4
dynamic_cfg.mu = 0.6
dynamic_cfg.restitution = 0.35

# -----------------------------
# Open square box without lid
# -----------------------------
# Newton convention: x, y, z
box_half_width = 4.0
wall_thickness = 0.3
wall_height = 4.0
floor_thickness = 0.25

# Static world body is usually -1
static_body = -1

# Floor
builder.add_shape_box(
    static_body,
    hx=box_half_width,
    hy=box_half_width,
    hz=floor_thickness * 0.5,
    xform=wp.transform(
        p=wp.vec3(0.0, 0.0, floor_thickness * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# +X wall
builder.add_shape_box(
    static_body,
    hx=wall_thickness * 0.5,
    hy=box_half_width,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(box_half_width + wall_thickness * 0.5, 0.0, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# -X wall
builder.add_shape_box(
    static_body,
    hx=wall_thickness * 0.5,
    hy=box_half_width,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(-box_half_width - wall_thickness * 0.5, 0.0, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# +Y wall
builder.add_shape_box(
    static_body,
    hx=box_half_width,
    hy=wall_thickness * 0.5,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(0.0, box_half_width + wall_thickness * 0.5, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# -Y wall
builder.add_shape_box(
    static_body,
    hx=box_half_width,
    hy=wall_thickness * 0.5,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(0.0, -box_half_width - wall_thickness * 0.5, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# -----------------------------
# Random rotations
# -----------------------------
rng = np.random.default_rng(42)

def random_quat():
    axis = rng.normal(size=3)
    axis = axis / np.linalg.norm(axis)
    angle = float(rng.uniform(0.0, 2.0 * math.pi))
    return wp.quat_from_axis_angle(
        wp.vec3(float(axis[0]), float(axis[1]), float(axis[2])),
        angle,
    )

# -----------------------------
# Five shapes poured from z=10 to z=30
# -----------------------------
z_positions = np.linspace(10.0, 30.0, 5)

# Put them above the box opening with slight XY offsets
xy_positions = [
    (-0.8, -0.6),
    ( 0.7, -0.4),
    ( 0.0,  0.5),
    (-0.5,  0.7),
    ( 0.6,  0.6),
]

# Sphere
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[0][0], xy_positions[0][1], float(z_positions[0])),
        q=random_quat(),
    ),
    label="sphere",
)
builder.add_shape_sphere(body, radius=0.5, cfg=dynamic_cfg)

# Box
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[1][0], xy_positions[1][1], float(z_positions[1])),
        q=random_quat(),
    ),
    label="box",
)
builder.add_shape_box(body, hx=0.5, hy=0.3, hz=0.8, cfg=dynamic_cfg)

# Cone
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[2][0], xy_positions[2][1], float(z_positions[2])),
        q=random_quat(),
    ),
    label="cone",
)
builder.add_shape_cone(body, radius=0.4, half_height=0.6, cfg=dynamic_cfg)

# Cylinder
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[3][0], xy_positions[3][1], float(z_positions[3])),
        q=random_quat(),
    ),
    label="cylinder",
)
builder.add_shape_cylinder(body, radius=0.35, half_height=0.5, cfg=dynamic_cfg)

# Capsule
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[4][0], xy_positions[4][1], float(z_positions[4])),
        q=random_quat(),
    ),
    label="capsule",
)
builder.add_shape_capsule(body, radius=0.3, half_height=0.5, cfg=dynamic_cfg)

add_particle(
    builder,
    '/home/bg/Developer/particle-pack-generation/data/toy_data_GOH_6_250/tmp/1944.obj',
    position=(0.0, 0.0, 40.0),
    orientation=(0, 0, 0, 0),
    scale=0.01,
    dynamic_cfg=dynamic_cfg
)

# -----------------------------
# Finalise model
# -----------------------------
model = builder.finalize()

collision_pipeline = newton.CollisionPipeline(
    model,
    broad_phase="sap",
)

solver = newton.solvers.SolverXPBD(
    model,
    iterations=12,
    rigid_contact_relaxation=0.8,
)

state_0 = model.state()
state_1 = model.state()
control = model.control()

newton.eval_fk(
    model,
    model.joint_q,
    model.joint_qd,
    state_0,
)

contacts = collision_pipeline.contacts()

# -----------------------------
# Viewer setup
# -----------------------------
viewer.set_model(model)

viewer.set_camera(
    pos=wp.vec3(10.0, -14.0, 10.0),
    pitch=-30.0,
    yaw=135.0,
)



In [18]:
# -----------------------------
# Simulation loop
# -----------------------------
for frame in tqdm(range(num_frames)):
    for _ in range(sim_substeps):
        state_0.clear_forces()

        contacts = model.collide(
            state_0,
            collision_pipeline=collision_pipeline,
        )

        viewer.apply_forces(state_0)

        solver.step(
            state_0,
            state_1,
            control,
            contacts,
            sim_dt,
        )

        state_0, state_1 = state_1, state_0

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()

    sim_time += frame_dt

100%|██████████| 600/600 [00:09<00:00, 61.12it/s]


In [19]:
viewer.show_notebook()

HTML(value='<div id="7a11ef98-9b90-4c14-a6a2-ab5f21d9e19b"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

In [ ]:
@dataclass
class SimulationSettings:
    frame_dt: float = 1.0 / 60.0
    sim_substeps: int = 10
    sim_time: float = 0.0
    num_frames: int = 600

    def __post_init__(self):
        self.sim_dt = self.frame_dt / self.sim_substeps

sim_settings = SimulationSettings()

SimulationSettings(frame_dt=0.016666666666666666, sim_substeps=10, sim_time=0.0, num_frames=600)


In [ ]:
class Simulator:
    def __init__(
        self,
        static_cfg: newton.ModelBuilder.ShapeConfig = None,
        dynamic_cfg: newton.ModelBuilder.ShapeConfig = None,
        viewer = None
    ):
        self.builder = newton.ModelBuilder()
        self.static_cfg = static_cfg
        self.dynamic_cfg = dynamic_cfg
        self.init_default_cfg()

        self.viewer = viewer
    
    def init_default_cfg(
        self
    ):
        if self.static_cfg is None:
            self.static_cfg = newton.ModelBuilder.ShapeConfig()
            self.static_cfg.ke = 1.0e5
            self.static_cfg.kd = 1.0e-4
            self.static_cfg.mu = 0.8
            self.static_cfg.restitution = 0.2
        if self.dynamic_cfg is None:
            self.dynamic_cfg = newton.ModelBuilder.ShapeConfig()
            self.dynamic_cfg.density = 1000.0
            self.dynamic_cfg.ke = 1.0e5
            self.dynamic_cfg.kd = 1.0e-4
            self.dynamic_cfg.mu = 0.6
            self.dynamic_cfg.restitution = 0.35
        
    def build_cubic_container(self, h, w, l):
        pass

    def build_cylindarical_container(self, radius, height):
        pass

    def add_particle(self):
        pass
    
    def add_particles(self,):
        pass

    def simulate(self,):
        pass


In [ ]:
# -----------------------------
# Build scene
# -----------------------------
builder = newton.ModelBuilder()

# -----------------------------
# Open square box without lid
# -----------------------------
# Newton convention: x, y, z
box_half_width = 4.0
wall_thickness = 0.3
wall_height = 4.0
floor_thickness = 0.25

# Static world body is usually -1
static_body = -1

# Floor
builder.add_shape_box(
    static_body,
    hx=box_half_width,
    hy=box_half_width,
    hz=floor_thickness * 0.5,
    xform=wp.transform(
        p=wp.vec3(0.0, 0.0, floor_thickness * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# +X wall
builder.add_shape_box(
    static_body,
    hx=wall_thickness * 0.5,
    hy=box_half_width,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(box_half_width + wall_thickness * 0.5, 0.0, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# -X wall
builder.add_shape_box(
    static_body,
    hx=wall_thickness * 0.5,
    hy=box_half_width,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(-box_half_width - wall_thickness * 0.5, 0.0, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# +Y wall
builder.add_shape_box(
    static_body,
    hx=box_half_width,
    hy=wall_thickness * 0.5,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(0.0, box_half_width + wall_thickness * 0.5, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# -Y wall
builder.add_shape_box(
    static_body,
    hx=box_half_width,
    hy=wall_thickness * 0.5,
    hz=wall_height * 0.5,
    xform=wp.transform(
        p=wp.vec3(0.0, -box_half_width - wall_thickness * 0.5, wall_height * 0.5),
        q=wp.quat_identity(),
    ),
    cfg=dynamic_cfg,
)

# -----------------------------
# Random rotations
# -----------------------------
rng = np.random.default_rng(42)

def random_quat():
    axis = rng.normal(size=3)
    axis = axis / np.linalg.norm(axis)
    angle = float(rng.uniform(0.0, 2.0 * math.pi))
    return wp.quat_from_axis_angle(
        wp.vec3(float(axis[0]), float(axis[1]), float(axis[2])),
        angle,
    )

# -----------------------------
# Five shapes poured from z=10 to z=30
# -----------------------------
z_positions = np.linspace(10.0, 30.0, 5)

# Put them above the box opening with slight XY offsets
xy_positions = [
    (-0.8, -0.6),
    ( 0.7, -0.4),
    ( 0.0,  0.5),
    (-0.5,  0.7),
    ( 0.6,  0.6),
]

# Sphere
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[0][0], xy_positions[0][1], float(z_positions[0])),
        q=random_quat(),
    ),
    label="sphere",
)
builder.add_shape_sphere(body, radius=0.5, cfg=dynamic_cfg)

# Box
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[1][0], xy_positions[1][1], float(z_positions[1])),
        q=random_quat(),
    ),
    label="box",
)
builder.add_shape_box(body, hx=0.5, hy=0.3, hz=0.8, cfg=dynamic_cfg)

# Cone
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[2][0], xy_positions[2][1], float(z_positions[2])),
        q=random_quat(),
    ),
    label="cone",
)
builder.add_shape_cone(body, radius=0.4, half_height=0.6, cfg=dynamic_cfg)

# Cylinder
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[3][0], xy_positions[3][1], float(z_positions[3])),
        q=random_quat(),
    ),
    label="cylinder",
)
builder.add_shape_cylinder(body, radius=0.35, half_height=0.5, cfg=dynamic_cfg)

# Capsule
body = builder.add_body(
    xform=wp.transform(
        p=wp.vec3(xy_positions[4][0], xy_positions[4][1], float(z_positions[4])),
        q=random_quat(),
    ),
    label="capsule",
)
builder.add_shape_capsule(body, radius=0.3, half_height=0.5, cfg=dynamic_cfg)

# -----------------------------
# Finalise model
# -----------------------------
model = builder.finalize()

collision_pipeline = newton.CollisionPipeline(
    model,
    broad_phase="sap",
)

solver = newton.solvers.SolverXPBD(
    model,
    iterations=12,
    rigid_contact_relaxation=0.8,
)

state_0 = model.state()
state_1 = model.state()
control = model.control()

newton.eval_fk(
    model,
    model.joint_q,
    model.joint_qd,
    state_0,
)

contacts = collision_pipeline.contacts()

# -----------------------------
# Viewer setup
# -----------------------------
viewer.set_model(model)

viewer.set_camera(
    pos=wp.vec3(10.0, -14.0, 10.0),
    pitch=-30.0,
    yaw=135.0,
)



In [13]:
# -----------------------------
# Simulation loop
# -----------------------------
for frame in tqdm(range(num_frames)):
    for _ in range(sim_substeps):
        state_0.clear_forces()

        contacts = model.collide(
            state_0,
            collision_pipeline=collision_pipeline,
        )

        viewer.apply_forces(state_0)

        solver.step(
            state_0,
            state_1,
            control,
            contacts,
            sim_dt,
        )

        state_0, state_1 = state_1, state_0

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()

    sim_time += frame_dt

100%|██████████| 600/600 [00:08<00:00, 66.74it/s]


In [14]:
viewer.show_notebook()

HTML(value='<div id="4f9c81db-c12c-41d9-b4cc-839638d0891f"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…